# PeerBERT runner (Kaggle / Colab)
Runs the pipeline in order. Every step is restart-safe: re-run a cell after a disconnect.
Train on ONE GPU (Kaggle 2xT4: GPU 0). Do not reinstall torch.

In [ ]:
import os, subprocess, sys
REPO = "https://github.com/Wajahatali56/PeerBERT.git"
BRANCH = "claude/brave-dirac-qt3mdw"
WORK = "/kaggle/working" if os.path.exists("/kaggle") else "/content"
os.chdir(WORK)
if not os.path.exists("PeerBERT"):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO], check=True)
os.chdir(f"{WORK}/PeerBERT")
os.environ["PRAD_WORK_DIR"] = WORK
print(os.getcwd(), "-> outputs in", WORK)

In [ ]:
!pip install -q -r requirements.txt pytest
!python -m pytest -q tests

## 1. Inspection, audit and manifests

In [ ]:
!python scripts/00_inspect_datasets.py --include-semdetect
!python scripts/02_prepare_splits.py
!python scripts/02_prepare_splits.py --verify-only

## 2. PeerBERT (smoke test first, then 3 seeds)

In [ ]:
!python scripts/03_train_intellabs.py --smoke --gpu 0
!python scripts/04_train_semdetect.py --smoke --gpu 0

In [ ]:
!python scripts/03_train_intellabs.py --gpu 0
!python scripts/04_train_semdetect.py --gpu 0

In [ ]:
!python scripts/05_evaluate.py --dataset intellabs --gpu 0
!python scripts/05_evaluate.py --dataset semdetect --gpu 0

## 3. Baselines (heavy: run detector by detector if the session is short)

In [ ]:
!python scripts/06_run_baselines.py --dataset intellabs --gpu 0
!python scripts/06_run_baselines.py --dataset semdetect --gpu 0

## 4. Ablation, cross-dataset transfer, explainability, report

In [ ]:
!python scripts/07_run_ablation.py --dataset intellabs --gpu 0
!python scripts/07_run_ablation.py --dataset semdetect --gpu 0
!python scripts/08_cross_dataset.py --source intellabs --gpu 0
!python scripts/08_cross_dataset.py --source semdetect --gpu 0
!python scripts/10_explain.py --dataset intellabs --gpu 0
!python scripts/10_explain.py --dataset semdetect --gpu 0
!python scripts/09_build_report.py